In [1]:
import time
from itertools import product

import numpy as np
import pandas as pd
from tqdm import trange

from backend import NumpyBackend, CudaBackend, Backend
import model as model
from model import GPT, load_model
from cugpt import GPT as cuGPT

xp: np = None

In [2]:
from dataset import *

math_data = build_math_dataset(
    max_number=999,
    include_operations=["plus", "minus", "multiply", "divide"],
    seed=42,
    val_fraction=1.0,
    examples_per_operation=20_000,
)

x_data = math_data.x_val
vocab_size = len(math_data.vocab)

In [3]:
def causal_mask(seq_len: int):
    upper = xp.triu(xp.ones((seq_len, seq_len), dtype=xp.float32), k=1)
    mask = xp.where(upper > 0, -xp.inf, 0.0)
    return mask[xp.newaxis, xp.newaxis, :, :]

In [4]:
batch_size = 64
tests = 16

model_path = "best_model.safetensors"

model_config = dict(
    vocab_size=vocab_size,
    blocks=2,
    model_dim=128,
    n_heads=4,
)

In [5]:
backends: list[Backend] = [NumpyBackend(), CudaBackend()]

models = {
    "GPT": GPT,
    "cuGPT": cuGPT,
}

In [6]:
def configure_backend(backend: Backend):
    global xp

    xp = model.xp = backend.xp
    model.scatter_add = backend.scatter_add
    model.as_numpy = backend.as_numpy

In [7]:
def build_model(model_name: str):
    cls = models[model_name]
    instance = cls(**model_config)

    if model_name == "GPT":
        load_model(instance, model_path)
    elif model_name == "cuGPT":
        instance.load(model_path)

    return instance

In [8]:
def forward(model_name: str, instance, xb, mask):
    if model_name == "GPT":
        return instance(xb, mask=mask)
    elif model_name == "cuGPT":
        return instance.forward(xb)

    raise ValueError(f"Unknown model: {model_name}")

In [9]:
def make_batches():
    for start in range(0, len(x_data), batch_size):
        yield x_data[start : start + batch_size]

In [10]:
experiments = list(product(backends, models.keys()))

len(experiments)

4

In [11]:
results = []

for backend, model_name in experiments:
    configure_backend(backend)

    instance = build_model(model_name)

    batches = [xp.asarray(batch, dtype=xp.int32) for batch in make_batches()]

    masks = [causal_mask(xb.shape[1]) for xb in batches]

    for test_id in (
        pbar := trange(tests, desc=f"{model_name} / {type(backend).__name__}")
    ):
        started = time.perf_counter()

        for batch_id, (xb, mask) in enumerate(zip(batches, masks), 1):
            forward(model_name, instance, xb, mask)
            pbar.set_postfix(Batches=f"{batch_id}/{len(batches)}")

        elapsed = time.perf_counter() - started

        results.append(
            {
                "test": test_id,
                "backend": type(backend).__name__,
                "model": model_name,
                "elapsed_s": elapsed,
                "examples": len(x_data),
                "batches": len(batches),
                "batch_size": batch_size,
            }
        )

    del instance

cuGPT / CudaBackend: 100%|██████████| 16/16 [00:23<00:00,  1.46s/it, Batches=893/893]


In [12]:
df = pd.DataFrame(results)

df.head()

,test,backend,model,elapsed_s,examples,batches,batch_size
0,0,NumpyBackend,GPT,59.816671,57104,893,64
1,1,NumpyBackend,GPT,59.834700,57104,893,64
2,2,NumpyBackend,GPT,56.373520,57104,893,64
3,3,NumpyBackend,GPT,59.919588,57104,893,64
4,4,NumpyBackend,GPT,59.097928,57104,893,64


In [13]:
summary = df.groupby(["backend", "model"], as_index=False).agg(
    mean_s=("elapsed_s", "mean"),
    std_s=("elapsed_s", "std"),
    min_s=("elapsed_s", "min"),
    max_s=("elapsed_s", "max"),
)

summary["examples_per_s"] = df["examples"] / summary["mean_s"]

summary["batches_per_s"] = df["batches"] / summary["mean_s"]

summary.sort_values("mean_s")

,backend,model,mean_s,std_s,min_s,max_s,examples_per_s,batches_per_s
1,CudaBackend,cuGPT,1.463706,0.044677,1.401656,1.576108,39013.290790,610.095066
3,NumpyBackend,cuGPT,1.524076,0.042826,1.460358,1.635788,37467.941339,585.928685
0,CudaBackend,GPT,8.455667,0.065412,8.251388,8.524606,6753.340364,105.609641
2,NumpyBackend,GPT,61.330138,8.487225,55.985789,91.111389,931.091993,14.560541


In [14]:
comparison = summary.pivot(
    index="backend",
    columns="model",
    values="mean_s",
)

comparison

model,GPT,cuGPT
backend,,
CudaBackend,8.455667,1.463706
NumpyBackend,61.330138,1.524076


In [15]:
best_time = summary["mean_s"].min()

summary["speedup_vs_slowest"] = summary["mean_s"].max() / summary["mean_s"]

summary.sort_values("mean_s")

,backend,model,mean_s,std_s,min_s,max_s,examples_per_s,batches_per_s,speedup_vs_slowest
1,CudaBackend,cuGPT,1.463706,0.044677,1.401656,1.576108,39013.290790,610.095066,41.900576
3,NumpyBackend,cuGPT,1.524076,0.042826,1.460358,1.635788,37467.941339,585.928685,40.240859
0,CudaBackend,GPT,8.455667,0.065412,8.251388,8.524606,6753.340364,105.609641,7.253140
2,NumpyBackend,GPT,61.330138,8.487225,55.985789,91.111389,931.091993,14.560541,1.000000
